In [20]:
# %pip install -U langchain langchain-openai

from pathlib import Path
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI
from rich import print
from pydantic import SecretStr


In [ ]:
def print_response(response: AIMessage):
    print(f'Response id: {response.id}')
    if response.usage_metadata is not None:
        input_tokens = response.usage_metadata.get('input_tokens', 0)
        cached_tokens = response.usage_metadata.get('input_token_details', {}).get('cache_read', 0)
        output_tokens = response.usage_metadata.get('output_tokens', 0)
        reasoning_tokens = response.usage_metadata.get('output_token_details', {}).get('reasoning', 0)

        print(f'Input tokens: {input_tokens} ({cached_tokens} cached); Output tokens: {output_tokens} ({reasoning_tokens} reasoning)')

    print()
    print(f"{'-' * 20} [Output] {'-' * 20}")
    print(response.text)

In [22]:
openai_api_key = SecretStr(Path('openai-secret-key-ai-integrations-developers.txt').read_text(encoding='utf-8').strip())
openai_model = ChatOpenAI(model_name='gpt-5-nano', openai_api_key=openai_api_key, reasoning_effort="low")

In [23]:
messages = [
SystemMessage(content="You are a helpful assistant that translates English to French."),
HumanMessage(content="Translate the following English text to French: 'Hello, how are you?'")
]

In [25]:
response = openai_model.invoke(
    input= messages
)

In [28]:
print_response(response)

Response id: lc_run--01a07674-aabf-7050-aada-f00505f9c2a3-0

Input tokens: 36 (0 cached); Output tokens: 79 (64 reasoning)

-------------------- [Output] --------------------

Bonjour, comment ça va ?

In [30]:
from pydantic import BaseModel

class WorkshopBrief(BaseModel): 
    title:str
    audience: str
    duration_minutes:float
    key_takeaways: list[str]

openai_astructured_output_model = openai_model.with_structured_output(WorkshopBrief)

In [ ]:
response = openai_astructured_output_model.invoke(
    input=[
        SystemMessage("You are en expert event organizer"), 
        HumanMessage("Design a beginner-friendly Saturday workshop about balcony herb gardening")
    ]
)
print_response(response)

OpenAIConnectionError: Connection error.